# MPNet fixed-schedule recovery on A100

The first attempt stopped on nonfinite scaled gradients before holdout evaluation. This frozen numerical amendment adds bounded AMP backoff on the same batch, with all six trials restarted. Model, data, page order, learning rates, precision and validation selection remain fixed. The complete first two epochs must reproduce the original runs before the same untouched 453-page holdout can be evaluated.

### 1. Install the same pinned environment

In [ ]:
import os, subprocess, sys
from pathlib import Path
os.environ.update(USE_TF='0', USE_FLAX='0', WANDB_MODE='disabled', TOKENIZERS_PARALLELISM='false')
packages = ['torch==2.9.1','sentence-transformers==5.1.2','transformers==4.57.3','datasets==4.4.1','xgboost==3.1.2','scikit-learn==1.7.2','numpy==2.3.5','scipy==1.16.3','beautifulsoup4==4.14.2','accelerate==1.11.0']
for command in [[sys.executable,'-m','pip','uninstall','-y','torchvision','torchaudio'], [sys.executable,'-m','pip','install','-q'] + packages]:
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        if process.wait():
            raise RuntimeError('Environment setup failed')
print('Pinned environment installed. Model training uses fresh Python subprocesses.')


### 2. Verify the requested A100

In [ ]:
import subprocess, sys, json
code = "import torch,json; assert torch.cuda.is_available() and 'A100' in torch.cuda.get_device_name(); print(json.dumps({'gpu':torch.cuda.get_device_name(),'memory_GiB':torch.cuda.get_device_properties(0).total_memory/2**30}))"
print(subprocess.check_output([sys.executable,'-c',code],text=True))


## Data
### 3. Verify frozen inputs and prior checkpoints

In [ ]:
import tarfile, hashlib, json, shutil
from pathlib import Path
archive=Path('/content/supervised_schedule_recovery_inputs.tar.gz')
assert hashlib.sha256(archive.read_bytes()).hexdigest()=='2055e0ea584e861e48c065d40eb10099175331c89b8653e115f1cf223c694ffb'
output=Path('/content/mpnet_supervised_schedule_recovery')
assert not output.exists(), 'Recovery requires a fresh output directory'
with tarfile.open(archive) as bundle:
    bundle.extractall('/content',filter='data')
for seed in [17,42,73]:
    shutil.copy2(Path('/content/mpnet_supervised_schedule/prior')/f'finetuned_{seed}'/'checkpoint.pt',output/'prior'/f'finetuned_{seed}'/'checkpoint.pt')
protocol=json.loads((output/'protocol.json').read_text())
assert protocol['retry_amp_overflow'] is True
for name, expected in protocol['files'].items():
    with (output/name).open('rb') as stream:
        assert hashlib.file_digest(stream,'sha256').hexdigest()==expected,name
print('Frozen numerical amendment:',protocol['amp_overflow_policy'])
print('Frozen counts:',protocol['counts'])
print('Frozen windows:',protocol['windows'])
print('Frozen at:',protocol['frozen_at'])


### 4. Reproduce the prior packages on validation

In [ ]:
import subprocess, sys, os

def run_stage(stage, command):
    with Path(f'/content/schedule_recovery_{stage}.log').open('w') as log:
        with subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,env=dict(os.environ,USE_TF='0',USE_FLAX='0',WANDB_MODE='disabled')) as process:
            for line in process.stdout:
                log.write(line); log.flush(); print(line,end='',flush=True)
            if process.wait(): raise RuntimeError(stage+' failed')

run_stage('amp_smoke',[sys.executable,'-c',"import sys,torch\nsys.path.insert(0,'/content/mpnet_supervised_schedule_recovery')\nfrom supervised_pilot import amp_step\nlayer=torch.nn.Linear(1,1,bias=False).cuda(); layer.weight.data.fill_(1.)\noptimizer=torch.optim.SGD(layer.parameters(),lr=.1)\nscaler=torch.amp.GradScaler('cuda',init_scale=65536.)\nwith torch.autocast('cuda',dtype=torch.float16): loss=layer(torch.ones(1,1,device='cuda')).sum()\nnorm,scale,retries=amp_step(loss,optimizer,scaler,True)\nassert len(retries)==1 and scale==32768. and norm==1.\ntorch.testing.assert_close(layer.weight,torch.tensor([[.9000001]],device='cuda'))\nprint('A100 forced AMP overflow recovered with one finite update and no invalid mutation')\n"])

for seed in [17,42,73]:
    folder=output/'prior'/f'finetuned_{seed}'
    run_stage(f'prior_replay_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'replay','--package',str(folder),'--input',str(output/'validation.jsonl'),'--expected',str(folder/'validation_predictions.jsonl'),'--output',str(folder/'package_replay.json')])


### 5. Train all six trials for six fixed epochs
Only training and validation influence checkpoint selection. Diagnostics record gradients, optimizer steps, and selected training windows.

In [ ]:
for seed in [17,42,73]:
    report=json.loads((output/'prior'/f'finetuned_{seed}'/'package_replay.json').read_text())
    assert report['n']==192 and report['identical_decisions'] and report['max_absolute_score_error']<=2e-6
run_stage('train',[sys.executable,'-u',str(output/'supervised_pilot.py'),'train','--output',str(output)])


### 6. Verify prefix reproduction and new inference packages

In [ ]:
assert (output/'selection_frozen.json').exists()
run_stage('prefix',[sys.executable,'-u',str(output/'supervised_schedule.py'),'prefix-check','--output',str(output)])
for seed in [17,42,73]:
    folder=output/f'finetuned_{seed}'
    run_stage(f'export_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'export','--package',str(folder),'--assets',str(output/'prior/finetuned_17/inference_assets')])
    run_stage(f'new_replay_{seed}',[sys.executable,'-u',str(output/'mpnet_inference.py'),'replay','--package',str(folder),'--input',str(output/'validation.jsonl'),'--expected',str(folder/'validation_predictions.jsonl'),'--output',str(folder/'package_replay.json')])


### 7. Evaluate the fresh holdout
All twelve old/new checkpoints and thresholds must be frozen. The comparison includes every seed.

In [ ]:
run_stage('evaluate',[sys.executable,'-u',str(output/'supervised_schedule.py'),'evaluate','--output',str(output)])


## Results
### 8. Compare schedules without selecting a winning seed

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
result=json.loads((output/'holdout_results.json').read_text())
assert len(result['summaries'])==len(result['prior_summaries'])==6
for seed in [17,42,73]:
    for label,key in [('Prior schedule','prior_summaries'),('Six epochs','summaries')]:
        row=result[key][f'finetuned_{seed}']
        print(f"{seed} {label}: AUC={row['roc_auc']:.4f}, recall={row['recall']['value']:.2%}, FPR={row['fpr']['value']:.2%}")
fig,axes=plt.subplots(1,3,figsize=(12,3.7),layout='constrained')
for ax,metric,title in zip(axes,['roc_auc','recall','fpr'],['ROC-AUC','Attack recall','False-positive rate']):
    for offset,(label,key,color,marker) in zip([-.12,.12],[('Prior schedule','prior_summaries','#2364aa','o'),('Six epochs','summaries','#b24c16','s')]):
        values=[result[key][f'finetuned_{seed}'][metric] for seed in [17,42,73]]
        values=[v['value'] if isinstance(v,dict) else v for v in values]
        ax.scatter([i+offset for i in range(3)],values,label=label,color=color,marker=marker)
    ax.set(title=title,xticks=range(3),xticklabels=['17','42','73'],xlabel='Training seed',ylim=(0,1))
    if metric!='roc_auc': ax.yaxis.set_major_formatter(PercentFormatter(1))
    ax.grid(axis='y',alpha=.2)
axes[0].legend()
axes[2].axhline(.05,color='#555555',ls='--',lw=1,label='Validation budget (5%)')
axes[2].legend(fontsize=8)
fig.suptitle('Fresh BrowseSafe holdout: 453 pages; all validation-frozen operating points')
fig.savefig(output/'schedule_comparison.png',dpi=180)
plt.show()
print('Nominal paired intervals and independent replay are computed after artifact retrieval.')


## Takeaways
Report recall together with actual false positives for every seed. A validation constraint is not a population guarantee. This is a BrowseSafe-only research experiment; unknown template relationships and source-label uncertainty remain. The 453-page holdout is consumed once evaluated.

Download all manifest files and logs before stopping the A100 session. Prior checkpoint files can be restored from their verified existing local copies. Avoid duplicating large result archives on the Mac.